# CVAE for the design of DD laminates
## Double-Double Laminates  [a / -a / b / -b]_X

**Optimization problem :**
$$
\min_{a,b,X}\; m(X) \quad \text{s.c.}\;
E_x \geq 30\,\text{GPa},\quad
N_{x,cr} \geq 1.5 \cdot |N_x| = 300\,\text{N/mm},\quad
4 \leq 4X \leq 64
$$

**Pipeline :**
1. Generating the DD dataset  
2. Training the CVAE  
3. CVAE-guided optimization  
4. Comparison with exhaustive search  


In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from skimpy import skim

from config import Config, FEATURE_NAMES
from material_registry import print_summary, get_mat, MATERIAL_KEYS
from materials import PlyProperties, dd_properties, dd_sequence
from dataset_dd import (generate_dd_dataset, save_dataset, load_dataset,
                        make_condition_vector, denormalise_features, 
                        _normalise_conditions, _denormalise_conditions)
from cvae import CVAE, count_parameters
from train import train as train_cvae, load_model
from lookup_table import DDLookupTable
from optimize import DDProblem, solve_dd_problem, brute_force_grid
from evaluate import run_full_evaluation
from visualize import (plot_training_curves, plot_lp_parity,
                    plot_miki_coverage, plot_pareto_front, 
                    plot_design_space, plot_material_comparison,
                    plot_miki_trajectory, plot_polar_stiffness,
                    plot_convergence_mass_penalty, plot_top_materials_projection,plot_miki_trajectory,plot_lp_generation_quality,plot_feasibility_heatmap)



%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Imports OK  |  PyTorch', torch.__version__)

## 1. Configuration

In [ ]:
cfg = Config()

# Display available materials
print("MATERIALS CATALOG")
print_summary()

# Optimization problem
problem = DDProblem(
    Nx_applied  = -200.0,
    plate_a     =  500.0,
    plate_b     =  500.0,
    Ex_min_GPa  =   30.0,
    SF_buckling =    1.5,
    allowed_mats = None # None = The CVAE can search among the 12 materials
)
print(f'\nNCR required : {problem.Ncr_required} N/mm')
print(f'Ex min req : {problem.Ex_min_GPa} GPa')

## 2. CLT Verification – Example of DD Laminate

In [ ]:
# test with the winner: IM7/977-3
mat_opt = get_mat('im7').ply_props
# Example : [32.7/-32.7/45/-45]_11  →  44 plis (X=11)
a_ex, b_ex, X_ex = 32.7, 45.0, 11
props = dd_properties(a_ex, b_ex, X_ex, mat_opt, problem.plate_a, problem.plate_b)

print(f"Laminate [{a_ex}°/-{a_ex}°/{b_ex}°/-{b_ex}°]_{X_ex} in IM7/977-3")
print(f'  n_plies = {4*X_ex}   h = {props["h_mm"]:.2f} mm')
print(f'  Ex      = {props["Ex_MPa"]/1e3:.2f} GPa  '
      f'(req ≥ {problem.Ex_min_GPa} GPa)  '
      + ('OK' if props["Ex_MPa"]/1e3 >= problem.Ex_min_GPa else '✗'))
print(f'  Ncr     = {props["Ncr_Npmm"]:.2f} N/mm  '
      f'(req ≥ {problem.Ncr_required:.0f} N/mm)  '
      + ('OK' if props["Ncr_Npmm"] >= problem.Ncr_required else '✗'))
print(f'  Mass   = {mat_opt.rho * problem.plate_a * problem.plate_b * props["h_mm"] * 1e3:.2f} g')

### Map Ex and Ncr in the space (a, b) for X=14

In [ ]:
# The design space remains specific to a given material for visualization
plot_design_space(X=11, mat=mat_opt, problem=problem, cfg=cfg, n_grid=60)

## 3. Generating the dataset

In [ ]:
import os; os.makedirs('data', exist_ok=True); os.makedirs('models', exist_ok=True)

if os.path.exists(cfg.data_path):
    df = load_dataset(cfg.data_path)
    print('Dataset loaded from disk.')
else:
    df = generate_dd_dataset(cfg, seed=cfg.seed)
    save_dataset(df, cfg.data_path)
    
print("Info Dataset")
print(df.info())

print("\n Breakdown of Materials")
print(df['mat_key'].value_counts(normalize=True).round(3) * 100)

skim(df)

plt.figure(figsize=(12, 8))
# Remove the one-hot encoding from the heatmap to keep it readable
cols_to_corr = [c for c in df.select_dtypes(include=np.number).columns if not c.startswith('mat_')]
sns.heatmap(df[cols_to_corr].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation between variables (excluding one-hot)")
plt.show()

In [ ]:
plot_miki_coverage(df_train=df)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
lp_raw_cols = [f'f_{n}_raw' for n in FEATURE_NAMES]
for i, (col, name) in enumerate(zip(lp_raw_cols, FEATURE_NAMES)):
    ax = axes[i // 3][i % 3]
    ax.hist(df[col], bins=80, color='steelblue', edgecolor='none', alpha=0.8)
    ax.axvline(0, color='k', ls='--', lw=0.8)
    ax.set_title(name); ax.set_xlabel('Value')
fig.suptitle('Distribution of rolling parameters (DD dataset)', fontsize=13)
fig.tight_layout(); plt.show()


## 4. Lookup table KD-tree

In [ ]:
LT_PATH = 'data/lookup_table.pkl'
if os.path.exists(LT_PATH):
    lt = DDLookupTable.load(LT_PATH)
else:
    lt = DDLookupTable.build(cfg, n_angles=360, verbose=True) 
    lt.save(LT_PATH)

# Verification: round trip (a,b) → LP → (a,b)
a_test, b_test, X_test = 30.0, 50.0, 10
from materials import compute_lamination_parameters
seq = dd_sequence(a_test, b_test, X_test)
lp  = compute_lamination_parameters(seq)
lp_vec = np.array([lp['a1'], lp['a2'], lp['d1'], lp['d2']])

a_rec, b_rec, res = lt.query_one(lp_vec, X=X_test)
print(f'Round trip (a,b) → LP → (a,b)  a={a_test}° → {a_rec:.2f}°   b={b_test}° → {b_rec:.2f}°   residual={res:.5f}')

In [ ]:
def test_normalization_integrity(cfg):
    print("=== STARTING NORMALIZATION INTEGRITY TEST ===\n")
    
    test_cases = {
        "Bornes Minimales": {
            "Nx": cfg.Nx_bounds[0], "Ny": cfg.Ny_bounds[0], "Nxy": cfg.Nxy_bounds[0],
            "plate_a": cfg.plate_a_bounds[0], "Ex_target": cfg.Ex_target_bounds[0], 
            "Nombre_de_plis": cfg.n_plies_bounds[0], "mat_key": "im7"
        },
        "Bornes Maximales": {
            "Nx": cfg.Nx_bounds[1], "Ny": cfg.Ny_bounds[1], "Nxy": cfg.Nxy_bounds[1],
            "plate_a": cfg.plate_a_bounds[1], "Ex_target": cfg.Ex_target_bounds[1], 
            "Nombre_de_plis": cfg.n_plies_bounds[1], "mat_key": "boron"
        },
        "Cas Zéro (Neutre)": {
            "Nx": 0.0, "Ny": 0.0, "Nxy": 0.0,
            "plate_a": (cfg.plate_a_bounds[0] + cfg.plate_a_bounds[1])/2, 
            "Ex_target": 50.0, "Nombre_de_plis": 24.0, "mat_key": "carbon"
        }
    }

    names = ["Nx", "Ny", "Nxy", "a", "b", "buckl", "fpf", "Ex", "n_plies"] + [f"m_{k}" for k in MATERIAL_KEYS]

    for name, params in test_cases.items():
        raw_val = make_condition_vector(**params, normalize=False, cfg=cfg)
        norm_val = _normalise_conditions(raw_val.copy(), cfg)
        recovered_val = _denormalise_conditions(norm_val.copy(), cfg)
        
        error = np.abs(raw_val - recovered_val)
        max_err = np.max(error)
        
        print(f"--- Test: {name} ---")
        if max_err < 1e-4:
            print(f"OK SUCCESS (Max error: {max_err:.2e})")
        else:
            print(f"KO FAILURE (Max error: {max_err:.2e})")

test_normalization_integrity(cfg)

## 5. CVAE Training

In [ ]:
if os.path.exists(cfg.model_path):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model  = load_model(cfg.model_path, cfg, device)
    print(f'Model loaded ({count_parameters(model):,} parameters).')
else:
    print(f'Training on {len(df)} samples ...')
    model = train_cvae(cfg) 

In [ ]:
if os.path.exists(cfg.log_dir + 'training_log.csv'):
    plot_training_curves(cfg.log_dir + 'training_log.csv')
    plot_convergence_mass_penalty(cfg.log_dir + 'training_log.csv')


## 6. Quality of the reconstruction

In [ ]:
plot_lp_parity(model, df, cfg)


## 7. Full Evaluation Report

In [ ]:
from torch.utils.data import random_split
from dataset_dd import DDLaminateDataset

ds       = DDLaminateDataset(df)
n_val    = max(100, int(0.10 * len(ds)))
n_train  = len(ds) - n_val
_, ds_val = random_split(ds, [n_train, n_val],
                          generator=torch.Generator().manual_seed(cfg.seed))
df_val = df.iloc[list(ds_val.indices)]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
report = run_full_evaluation(model, df, df_val, problem, cfg, lt,
                              device=device, verbose=True)

df['mass_kg'] = (df['plate_a'] * df['plate_b'] * 1e-6) * (df['h_mm'] * 1e-3) * df['rho']
# Display of the material comparison chart
plot_material_comparison(df,problem  )



In [ ]:
# Selecting a material and a target X to visualize the model's behavior
target_mat = "im7" 
target_X = 11

# View the effect of temperature on the feasibility of Miki
plot_miki_trajectory(model, problem, cfg, lt, 
                     temperatures=[0.3, 0.7, 1.0, 1.5, 2.0], 
                     mat_key=target_mat, X=target_X)

# Check whether the generated LP files match the actual distribution
plot_lp_generation_quality(model, df, problem, cfg, 
                           mat_key=target_mat, X=target_X)

## 8. Optimization

In [ ]:
results = solve_dd_problem(
    cfg=cfg, problem=problem, model=model, 
    run_brute_force=True,
    n_samples_per_X=5_000,
    temperature=1.0,
    lookup_table=lt,
)

In [ ]:
df_bf   = results.get('brute_force', pd.DataFrame())
df_cvae = results.get('cvae',        pd.DataFrame())

if not df_bf.empty:
    plot_pareto_front(df_bf, problem)
    # Full 2D rendering of the best materials
    plot_top_materials_projection(df_bf, problem, top_n=6)
    #  Feasibility Heatmap
    plot_feasibility_heatmap(df_bf, cfg, metric="n_feasible")
if not df_cvae.empty:
# Compare LP generation quality on the CVAE
    target_mat = df_cvae.iloc[0]['mat_key']
    target_X = int(df_cvae.iloc[0]['X'])
    plot_lp_generation_quality(model, df, problem, cfg, mat_key=target_mat, X=target_X)

## 9. Pareto Front and Comparison

In [ ]:
df_bf   = results.get('brute_force', pd.DataFrame())
df_cvae = results.get('cvae',        pd.DataFrame())

if not df_bf.empty:
    plot_pareto_front(df_bf, problem)

if not df_cvae.empty and not df_bf.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(df_bf['Ncr_Npmm'],   df_bf['Ex_GPa'],
               s=8, alpha=0.3, label='Brute force', color='steelblue')
    ax.scatter(df_cvae['Ncr_Npmm'], df_cvae['Ex_GPa'],
               s=30, alpha=0.7, marker='*', label='CVAE', color='tomato')
    ax.axvline(problem.Ncr_required, color='k', ls='--', lw=1, label='Ncr req.')
    ax.axhline(problem.Ex_min_GPa,  color='gray', ls=':', lw=1, label='Ex req.')
    ax.set_xlabel('Ncr [N/mm]'); ax.set_ylabel('Ex [GPa]')
    ax.set_title('Brute-force vs. CVAE — Space for feasible designs (all materials)')
    ax.legend(); plt.tight_layout(); plt.show()

## 10. Angular Sensitivity for Optimal Design

In [ ]:
best = results.get('best', {})
if best:
    X_opt = best['X']
    mat_opt_key = best['mat_key']
    mat_opt = get_mat(mat_opt_key).ply_props # Dynamic Recovery
    
    print(f"OPTIMAL MULTI-MATERIAL DESIGN :")
    print(f"  Material = {mat_opt_key}")
    print(f"  SSequence = [{best['a_deg']}° / -{best['a_deg']}° / {best['b_deg']}° / -{best['b_deg']}°]_{X_opt}")
    print(f"  Ex = {best['Ex_GPa']} GPa   Ncr = {best['Ncr_Npmm']} N/mm")
    print(f"  Mass = {best['mass_kg']*1e3:.2f} g")

    # Sensitivity map around the optimal point using the correct material
    a_range = np.linspace(max(0, best['a_deg'] - 15), best['a_deg'] + 15, 50)
    b_range = np.linspace(max(0, best['b_deg'] - 15), best['b_deg'] + 15, 50)
    AA, BB = np.meshgrid(a_range, b_range)
    Ncr_map = np.zeros_like(AA)
    Ex_map  = np.zeros_like(AA)
    for i in range(len(a_range)):
        for j in range(len(b_range)):
            p = dd_properties(AA[j,i], BB[j,i], int(X_opt), mat_opt, problem.plate_a, problem.plate_b)
            Ex_map[j,i]  = p['Ex_MPa'] / 1e3
            Ncr_map[j,i] = p['Ncr_Npmm']

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, data, title, req in zip(
        axes, [Ex_map, Ncr_map],
        ['Ex [GPa]', 'Ncr [N/mm]'], [problem.Ex_min_GPa, problem.Ncr_required]
    ):
        im = ax.pcolormesh(AA, BB, data, cmap='RdYlGn', shading='auto')
        ax.contour(AA, BB, data, levels=[req], colors='blue', linewidths=2)
        plt.colorbar(im, ax=ax)
        ax.scatter([best['a_deg']], [best['b_deg']],
                    marker='*', s=200, color='k', zorder=5, label='Opt.')
        ax.set_xlabel('a [°]'); ax.set_ylabel('b [°]')
        ax.set_title(f'{title}  (X={X_opt}, Mat={mat_opt_key})')
        ax.legend()
    fig.suptitle(f'Sensitivity regarding optimal design ({mat_opt_key})', fontsize=13)
    plt.tight_layout(); plt.show()

## 11. Effect of Temperature on Reproduction

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.eval()

target_mat = results.get('best', {}).get('mat_key', 'IM7/977-3')
target_X = results.get('best', {}).get('X', 11)

for ax, T in zip(axes, [0.3, 1.0, 2.0]):
    cond_norm = make_condition_vector(
        Nx=-problem.Ncr_required, plate_a=problem.plate_a,
        plate_b=problem.plate_b, Ex_target=problem.Ex_min_GPa,
        Nombre_de_plis=target_X * 4, mat_key=target_mat, # Fix ici
        normalize=True, cfg=cfg,
    )
    ct = torch.tensor(cond_norm, dtype=torch.float32, device=device)
    with torch.no_grad():
        lp_norm = model.generate(ct, n_samples=2000, temperature=T).cpu().numpy()
    lp_raw = denormalise_features(lp_norm)
    angles, _ = lt.query(lp_raw, X=target_X, k=1)

    ax.scatter(angles[:, 0], angles[:, 1], s=4, alpha=0.4)
    if results.get('best'):
        ax.scatter([results['best']['a_deg']], [results['best']['b_deg']],
                    marker='*', s=200, color='r', zorder=5, label='Opt. BF')
    ax.set_xlabel('a [°]'); ax.set_ylabel('b [°]')
    ax.set_title(f'T = {T}  —  dispersion')
    ax.set_xlim(0, 90); ax.set_ylim(0, 90)
    ax.legend(fontsize=8)
plt.suptitle(f'Effect of temperature (X={target_X}, Mat={target_mat})', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
best = results.get('best', {})
if best:
    X_opt = best['X']
    mat_opt_key = best['mat_key']
    mat_opt = get_mat(mat_opt_key).ply_props
    
    print(f"DESIGN OPTIMAL MULTI-MATÉRIAUX :")

    #  Plot the polar stiffness rose for the winning design
    plot_polar_stiffness(best_design=best, problem=problem, cfg=cfg)
